## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default
tf.enable_eager_execution()

### Collect Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd

In [0]:
data = pd.read_csv('/content/drive/My Drive/R6/prices.csv')

### Check all columns in the dataset

In [6]:
data.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Drop columns `date` and  `symbol`

In [0]:
data = data.drop(['date','symbol'],axis=1)

In [8]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [9]:
data.shape

(851264, 5)

In [0]:
data = data.iloc[:1000]

In [12]:
data

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0
5,115.510002,115.550003,114.500000,116.059998,1098000.0
6,116.459999,112.849998,112.589996,117.070000,949600.0
7,113.510002,114.379997,110.050003,115.029999,785300.0
8,113.330002,112.529999,111.919998,114.879997,1093700.0
9,113.660004,110.379997,109.870003,115.870003,1523500.0


### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
target = data['volume']
feature = data.drop(['volume'],axis=1)

In [0]:
x_train , x_test , y_train , y_test = train_test_split(feature, target, test_size = 0.2,random_state =7)

#### Convert Training and Test Data to numpy float32 arrays


In [0]:
import numpy as np
x_train = np.asarray(x_train).astype('float32')

In [0]:
x_test = np.array(x_test).astype('float32')

In [0]:
y_train = np.array(y_train).astype('float32')

In [0]:
y_test = np.array(y_test).astype('float32')

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import StandardScaler

In [0]:
from sklearn.preprocessing import Normalizer
# from sklearn import preprocessing
transformer = Normalizer()

x_train= transformer.transform(x_train)
x_test= transformer.transform(x_test)


In [23]:
x_train

array([[0.49873042, 0.5009242 , 0.49385527, 0.5064088 ],
       [0.49924964, 0.50368565, 0.49219242, 0.50477445],
       [0.5000853 , 0.50262463, 0.49324548, 0.5039762 ],
       ...,
       [0.5001929 , 0.49977148, 0.4976645 , 0.50236005],
       [0.50253594, 0.49852157, 0.496339  , 0.50257486],
       [0.5012146 , 0.49876764, 0.49595365, 0.50402856]], dtype=float32)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.random_normal(shape=[4,1])
b = tf.zeros(shape=[1])

In [26]:
W.numpy()

array([[-1.3827509],
       [-1.0280485],
       [-0.0399912],
       [-1.5159489]], dtype=float32)

2.Define a function to calculate prediction

In [0]:
def predection(x,W,b):
  return tf.add(tf.matmul(x_train,W),b)

3.Loss (Cost) Function [Mean square error]

In [30]:
def loss(x,y,W,b):
  return tf.reduce_mean(tf.square(y_train-predection(x,W,b))


SyntaxError: ignored

In [123]:
loss

<tf.Tensor 'Loss_1:0' shape=() dtype=float32>

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [0]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

In [128]:
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:x_train, y_:y_train})
    
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

ValueError: ignored

### Get the shapes and values of W and b

### Model Prediction on 1st Examples in Test Dataset

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

### Splitting the data into feature set and target set

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

### Model Training 

### Model Prediction

### Save the Model

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?